In [52]:
import math,time
import torch
import torch.nn as nn
import torch.nn.functional as F


In [53]:
#实现softmax
def softmax(x,dim=-1):
  max_val=torch.max(x,dim=-1,keepdim=True).values
  e=torch.exp(x-max_val)
  return e/torch.sum(e,dim=-1,keepdim=True)

In [54]:
#实现scaled_dot_product_attention
#q:B,H,t,dk
#k:B,H,t,dk
#v:B,H,t,dv
def scaled_dot_product_attention(q,k,v,mask=None):
  dk=q.size(-1)
  scores=q@k.transpose(-2,-1)
  if mask is not None:
    scores=scores.masked_fill(~mask,float("-inf"))
  att=softmax(scores)
  return att@v,att

In [55]:
#实现MultiHeadAttention
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,n_head,dropout=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.d_k=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(dropout)
  #转换形式（B,T,d_model）->(B,T,h,dk)->(B,h,T,dk)
  def split(self,x):
    B,T,_=x.shape
    x=x.view(B,T,self.h,self.d_k).transpose(1,2)
    return x

  def forward(self,q,k,v,mask=None):
    q=self.split(self.W_q(q))
    k=self.split(self.W_k(k))
    v=self.split(self.W_v(v))
    out,self.lastatt=scaled_dot_product_attention(q,k,v,mask)
    out=self.dropout(out)
    #(B,h,T,dk)->（B,T,d_model）
    B,h,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,h*dk)
    out=self.W_o(out)
    return out


In [56]:
#位置编码
class SinusoidialPE(nn.Module):
  def __init__(self,d_model,max_len=2048) -> None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)

  def forward(self,x):
    return x+self.pe[:x.size(1)]

In [57]:
#FFN
class FeedForwardNet(nn.Module):
  def __init__(self,d_model,d_dff,drop_out=0.01) -> None:
    super().__init__()
    self.net=nn.Sequential(nn.Linear(d_model,d_dff),nn.GELU(),nn.Linear(d_dff,d_model),nn.Dropout(drop_out))
  def forward(self,x):
    return self.net(x)

In [58]:
#Encoder
class EncoderLayer(nn.Module):
  def __init__(self,d_model,d_dff,n_head,drop_out=0.01) -> None:
    super().__init__()
    self.attn=MultiHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForwardNet(d_model,d_dff,drop_out)
    self.ln1=nn.LayerNorm(d_model)
    self.ln2=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)

  def forward(self,x,src_mask):
    x=x+self.dropout(self.attn(self.ln1(x),self.ln1(x),self.ln1(x),src_mask))
    x=x+self.dropout(self.ffn(self.ln2(x)))
    return x

In [59]:
#Dcoder
class DecoderLayer(nn.Module):
  def __init__(self,d_model,d_dff,n_head,drop_out=0.01) -> None:
    super().__init__()
    self.attn=MultiHeadAttention(d_model,n_head,drop_out)
    self.cross_attn=MultiHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForwardNet(d_model,d_dff,drop_out)
    self.ln1=nn.LayerNorm(d_model)
    self.ln2=nn.LayerNorm(d_model)
    self.ln3=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)

  def forward(self,x,memory,src_mask,tgt_mask):
    x=self.ln1(x)
    x=x+self.dropout(self.attn(x,x,x,tgt_mask))
    x=self.ln2(x)
    x=x+self.dropout(self.cross_attn(x,memory,memory,src_mask))
    x=self.ln3(x)
    x=x+self.dropout(self.ffn(x))
    return x



In [62]:
#Transfomer
class Transformer(nn.Module):
  def __init__(self,src_voc,tgt_voc,d_model=128,n_head=4,pad_id=0,d_dff=512,max_len=4096,n_layers=3,drop_out=0.01) -> None:
    super().__init__()
    self.padId,self.d_model=pad_id,d_model
    self.src_emb=nn.Embedding(src_voc,d_model,padding_idx=pad_id)
    self.tgt_emb=nn.Embedding(tgt_voc,d_model,padding_idx=pad_id)
    self.dropout=nn.Dropout(drop_out)
    self.PE=SinusoidialPE(d_model,max_len)
    self.enc_ln=nn.LayerNorm(d_model)
    self.dec_ln=nn.LayerNorm(d_model)
    self.out=nn.Linear(d_model,tgt_voc)
    self.enc=nn.ModuleList(EncoderLayer(d_model,d_dff,n_head,drop_out) for _ in range(n_layers))
    self.dec=nn.ModuleList(DecoderLayer(d_model,d_dff,n_head,drop_out) for _ in range(n_layers))

  def make_masks(self,src_voc,tgt_voc):
    src_mask=(src_voc!=self.padId)[:,None,None,:]
    Tt=tgt_voc.size(1)
    causal=torch.tril(torch.ones(Tt,Tt,dtype=bool,device=tgt_voc.device))
    tgt_mask=(tgt_voc!=self.padId)[:,None,None,:]&causal
    return src_mask,tgt_mask

  def encode(self,src,src_mask):
    x=self.dropout(self.PE(self.src_emb(src)*math.sqrt(self.d_model)))
    for layer in self.enc:x=layer(x,src_mask)
    return self.enc_ln(x)

  def decode(self,tgt,memory,src_mask,tgt_mask):
    x=self.dropout(self.PE(self.tgt_emb(tgt)*math.sqrt(self.d_model)))
    for layer in self.dec:x=layer(x,memory,src_mask,tgt_mask)
    return self.dec_ln(x)

  def forward(self,src,tgt):
    src_mask,tgt_mask=self.make_masks(src,tgt)
    out=self.decode(tgt,self.encode(src,src_mask),src_mask,tgt_mask)
    return self.out(out)
model=Transformer(20,20)
src=torch.randint(3,20,(2,8))
tgt=torch.randint(3,20,(2,9))
print(model(src, tgt).shape, "params:", sum(p.numel() for p in model.parameters()))

torch.Size([2, 9, 20]) params: 1396756
